# Stage 3 on Kaggle: LoRA fine-tune of Qwen3.5-0.8B on SentiMix (Hinglish sentiment)

Runs `python -m minigpt.lora --config configs/lora_kaggle.yaml`: lr sweep on val (1e-4, 2e-4, 5e-4),
then 3 seeds with the best lr, 2 epochs each, fp16 on a T4.

**Before running:**
1. *Settings → Accelerator → GPU T4 x2* (one GPU is used) and *Settings → Internet → On*.
2. Get the code here, either way:
   - **GitHub:** push the repo and put its URL in `REPO_URL` below, or
   - **Kaggle dataset:** on your PC run `git archive --format=zip -o dist/mini-gpt-src.zip HEAD`,
     upload `dist/mini-gpt-src.zip` as a Kaggle dataset, and attach it (*Add Input*).
3. *Run All*. At the end download `lora_results.zip` from the Output panel and unzip it into the project root.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, pathlib, subprocess, zipfile

REPO_URL = ""  # e.g. "https://github.com/<you>/mini-gpt.git"; leave empty to use the attached dataset zip
WORK = pathlib.Path("/kaggle/working/mini-gpt")
if WORK.exists():
    print("code already present")
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    zips = glob.glob("/kaggle/input/**/mini-gpt-src.zip", recursive=True)
    assert zips, "attach the mini-gpt-src.zip dataset or set REPO_URL"
    zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(WORK)
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
# Same library versions as the local run. The last two packages are optional fast kernels for
# Qwen3.5's linear-attention layers (Linux + Triton); if they fail to build, the slower PyTorch
# fallback is used and everything still works.
!pip install -q "transformers==5.17.0" "peft==0.21.1" accelerate ftfy python-dotenv
!pip install -q -e . --no-deps
!pip install -q flash-linear-attention causal-conv1d || echo 'fast kernels not installed: using the PyTorch fallback'

In [ ]:
# SentiMix only (downloads ~5 MB from the Hub); the 2.5 GB pre-training corpus is not needed here
!python -m minigpt.data --prepare --sentimix-only

In [ ]:
# Quick check that everything runs (60 tweets per split, 1 lr, a few minutes)
!python -m minigpt.lora --config configs/lora_kaggle.yaml --limit 60

In [ ]:
# The real run: lr sweep + 3 seeds
!python -m minigpt.lora --config configs/lora_kaggle.yaml

In [ ]:
out = pathlib.Path("/kaggle/working/lora_results.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["results/stage3/lora_kaggle", "checkpoints/stage3"]:
        for f in pathlib.Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f)
print(out, f"{out.stat().st_size / 1e6:.0f} MB")